# Week 14 Demo: Talking to the Gemini API

In this demo, we send a prompt from Python to Google's Gemini API and get the AI's answer back. We use the `requests` library to send the request over the internet, and we read the answer out of the JSON that comes back.

**Before you run this notebook**
- You need a Gemini API key from Week 14 Lab Part I.
- Open the lab folder in VS Code, create a `.venv`, and select it as the notebook kernel (the same steps as Week 13). The lab notebook walks you through it.
- Create a file named `config.py` in this folder with one line: `API_KEY = "your_key_here"`. The `.gitignore` in this folder already lists `config.py`, so Git will not upload it.

## Step 1: Setup

Install `requests` in the terminal with your `.venv` active, then save your requirements:

```
pip install requests ipykernel
pip freeze > requirements.txt
```

`json` comes with Python, so it is never installed. Next, we import both libraries and load the key from `config.py`. We print the length of the key to show it loaded. We never print the key itself.

In [ ]:
import requests
import json

try:
    from config import API_KEY
    print("Key loaded. Length:", len(API_KEY))
except ImportError:
    print("Could not load API_KEY from config.py. Check that config.py is in this folder and has the line: API_KEY = \"your_key_here\"")

## Step 2: The Endpoint and the Model

The **endpoint** is the web address we send our request to. Every Gemini model uses the same address. We choose the model inside the request.

Google adds new models and retires old ones often. Keep the model name in one variable, so changing models means changing one line. If a model is retired, you get a 404 error. If a model is busy, you get a 503 error, and you can wait a minute or try another model.

The key goes in the request **headers**, never in the web address. A web address can show up in printed output and error messages, and headers do not.

In [ ]:
url = "https://generativelanguage.googleapis.com/v1beta/interactions"
model_name = "gemini-3.5-flash-lite"
headers = {"x-goog-api-key": API_KEY}

**Optional: see which models your key can use.** The list is long, so we print only the names with "flash" in them.

In [ ]:
list_response = requests.get("https://generativelanguage.googleapis.com/v1beta/models",
                             headers=headers, params={"pageSize": 1000}, timeout=60)

if list_response.status_code == 200:
    for model in list_response.json()["models"]:
        if "flash" in model["name"]:
            print(model["name"])
else:
    print("Error listing models:", list_response.status_code)
    print(list_response.text)

## Step 3: The Payload (Our Order)

The **payload** is the data we send. Gemini expects a dictionary with two keys: `model` and `input`.

Our prompt uses the prompt engineering ideas from the first video: it gives the AI a role, an audience, a length, and a format.

In [ ]:
my_prompt = ("You are a patient kindergarten teacher. Explain what a variable is in programming "
             "to a 5-year-old in three short sentences. Use plain text with no formatting.")

payload = {"model": model_name, "input": my_prompt}

print(payload)

## Step 4: Send the Request

`requests.post()` sends the payload to the endpoint. `json=payload` turns our dictionary into JSON, and `timeout=60` stops waiting after 60 seconds, so a busy server cannot freeze the notebook.

The **status code** tells us what happened:
- **200**: OK.
- **400 or 403**: a problem with the key or the request.
- **404**: the model name is wrong, or the model was retired.
- **429**: too many requests. Wait a minute.
- **503**: the model is busy. Wait a minute or try another model.

In [ ]:
response = requests.post(url, headers=headers, json=payload, timeout=60)

print("Status code:", response.status_code)

if response.status_code != 200:
    print(response.text)   # Google's error message

## Step 5: Read the Response

`response.json()` turns the JSON into a Python dictionary. `json.dumps(..., indent=2)` prints it with indentation so we can see its structure.

In [ ]:
data = response.json()

print(json.dumps(data, indent=2))

The answer is inside `steps`, a list. The first step is the model's `thought` (its reasoning). The **last** step is the `model_output`, and its `content` list holds the text.

So the path is: `data["steps"]`, then `[-1]` for the last step, then `["content"]`, then `[0]` for the first item, then `["text"]`. Negative numbers count from the end of a list, so `[-1]` is always the last item, even if the number of steps changes.

In [ ]:
ai_text = data["steps"][-1]["content"][0]["text"]

print("--- AI Response ---")
print(ai_text)

## Step 6: A Reusable Function

That took several cells. A function puts the steps in one place, so asking a new question takes one line. The function gets everything it needs through its parameters, and it returns the AI's text, or an error message if something goes wrong.

In [ ]:
def ask_gemini(prompt, api_key, model):
    """Send a prompt to the Gemini API and return the AI's text, or an error message."""
    url = "https://generativelanguage.googleapis.com/v1beta/interactions"
    headers = {"x-goog-api-key": api_key}
    payload = {"model": model, "input": prompt}

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=60)
    except requests.exceptions.RequestException:
        return "Error: could not reach the Gemini API. Check your internet connection and try again."

    if response.status_code != 200:
        return f"Error {response.status_code}: {response.text}"

    data = response.json()
    return data["steps"][-1]["content"][0]["text"]

In [ ]:
print(ask_gemini("Give me three creative names for a pet rabbit. Plain text, one name per line.", API_KEY, model_name))

In [ ]:
print(ask_gemini("You are a friendly pet expert. In two sentences, explain what a rabbit needs on its first day home.", API_KEY, model_name))

## Before You Commit: Run From a Fresh Start

When your code works, run it once more from a fresh start before you commit. A fresh run proves the notebook does not depend on anything left over in Python's memory.

1. Click **Restart** in the notebook toolbar.
2. Click in the first code cell. Open the Command Palette (`Ctrl+Shift+P`, or `Cmd+Shift+P` on a Mac) and choose **Notebook: Execute Cell and Below**.
3. Check the numbers to the left of the code cells. They should read 1, 2, 3, and so on, in order, with no errors.

Each run sends the prompts again, so the AI's answers will change. Once you have written anything about an AI answer, do not run that cell again.

## Wrap-Up

- Run the same prompt twice and the wording changes. That is normal for AI, so your code should never depend on the exact words.
- AI answers can be wrong. Check anything that matters, the same way you check Copilot's suggestions.
- Keep your key in `config.py`, keep `config.py` in `.gitignore`, and never print the key.